In [ ]:
# import pandas as pd
# import numpy as np


# def load_dataset(name_dataset):
#     #leemos el dataset de futbol uruguayo
#     df = pd.read_csv(name_dataset)

#     #nos quedamos con las columnas que nos interesan para el clasificador
#     df = df.drop(columns=["full_time", "competition", "home_ident", "away_ident", "home_country", "away_country", "home_code", "away_code", "home_continent", "away_continent", "continent", "level"])

#     #convertimos las columnas a los tipos de datos correctos
#     df["date"] = pd.to_datetime(df["date"], errors="raise")
#     df["gh"] = pd.to_numeric(df["gh"], errors="raise").astype(int)
#     df["ga"] = pd.to_numeric(df["ga"], errors="raise").astype(int)

#     #ordenamos el dataset por fecha y por equipos, y eliminamos duplicados
#     df = (
#         df.drop_duplicates()
#         .sort_values(["date", "home", "away"], kind="stable")
#         .reset_index(drop=True)
#     )

#     #creamos la columna result, que es el resultado del partido, L si gana el local, V si gana el visitante y E si empatan
#     df["result"] = df.apply(
#         lambda row: "L" if row["gh"] > row["ga"] else ("V" if row["gh"] < row["ga"] else "E"), axis=1
#     )

#     return df

# def load_atributes(dataset: pd.DataFrame, date: pd.Timestamp, years_limit: int = 1, matches_limit: int = 5) -> pd.DataFrame:

#     def get_historial(date, team: str) -> pd.DataFrame:

#         #filtramos el dataset por el equipo y la fecha
#         df = dataset[(dataset["home"] == team) | (dataset["away"] == team)]
#         df = df[(date - pd.DateOffset(years=years_limit) < df["date"]) & (df["date"] < date)]

#         #eliminamos los que sean anteriores a un limite 
#         return df

#     def get_wins(dataset: pd.DataFrame, team: str) -> int:
#         #contamos las victorias del equipo
#         return len(dataset[(dataset["home"] == team) & (dataset["result"] == "L")]) + len(dataset[(dataset["away"] == team) & (dataset["result"] == "V")])

#     def comparar(win_reate_team1: float, win_reate_team2: float) -> str:
#         #comparamos las victorias de los equipos
#         if win_reate_team1 > win_reate_team2:
#             return "L"
#         elif win_reate_team1 < win_reate_team2:
#             return "V"
#         else:
#             return "E"

#     def get_condition(date, team: str) -> int:
#         #obtenemos el historial del equipo
#         df = get_historial(date, team)

#         #me quedo con los ultimos partidos del equipo (si los hay)
#         df = df.tail(matches_limit)

#         #TODO : quizas se puede hacer por puntos en vez de cantidad de victorias, pero por ahora lo dejamos asi
#         return (get_wins(df, team) / len(df) if len(df) > 0 else 0)
    

#     #creamos las nuevas columnas con el historial historico de los equipos
#     dataset["historial"] = comparar(dataset["home"].apply(get_historial, args=(dataset["date"], dataset["home"])), dataset["away"].apply(get_historial, args=(dataset["date"], dataset["away"])))

#     #creamos las nuevas columnas con las condiciones recientes de los equipos
#     dataset["condition_match"] = comparar(dataset["home"].apply(get_condition, args=(dataset["date"], dataset["home"])), dataset["away"].apply(get_condition, args=(dataset["date"], dataset["away"])))  
    
#     return dataset

    







In [1]:
import pandas as pd
import numpy as np





def load_dataset(name_dataset):
    #leemos el dataset de futbol uruguayo
    df = pd.read_csv(name_dataset)

    #nos quedamos con las columnas que nos interesan para el clasificador
    df = df.drop(columns=["full_time", "competition", "home_ident", "away_ident", "home_country", "away_country", "home_code", "away_code", "home_continent", "away_continent", "continent", "level"])

    #convertimos las columnas a los tipos de datos correctos
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    df["gh"] = pd.to_numeric(df["gh"], errors="raise").astype(int)
    df["ga"] = pd.to_numeric(df["ga"], errors="raise").astype(int)

    #ordenamos el dataset por fecha y por equipos, y eliminamos duplicados
    df = (
        df.drop_duplicates()
        .sort_values(["date", "home", "away"], kind="stable")
        .reset_index(drop=True)
    )

    #creamos la columna result, que es el resultado del partido, L si gana el local, V si gana el visitante y E si empatan
    df["result"] = df.apply(
        lambda row: "L" if row["gh"] > row["ga"] else ("V" if row["gh"] < row["ga"] else "E"), axis=1
    )

    return df

def load_attributes(
    dataset: pd.DataFrame,
    years_limit: int = 1,
    matches_limit: int = 5
) -> pd.DataFrame:

    dataset = dataset.copy()

    def get_historial(
        date: pd.Timestamp,
        team: str
    ) -> pd.DataFrame:

        fecha_inicio = date - pd.DateOffset(
            years=years_limit
        )

        historial = dataset[
            (
                (dataset["home"] == team)
                | (dataset["away"] == team)
            )
            & (dataset["date"] >= fecha_inicio)
            & (dataset["date"] < date)
        ]

        return historial.sort_values("date")

    def get_wins(
        historial: pd.DataFrame,
        team: str
    ) -> int:

        victorias_local = (
            (historial["home"] == team)
            & (historial["result"] == "L")
        ).sum()

        victorias_visitante = (
            (historial["away"] == team)
            & (historial["result"] == "V")
        ).sum()

        return int(
            victorias_local + victorias_visitante
        )

    def get_win_rate(
        historial: pd.DataFrame,
        team: str
    ) -> float:

        if len(historial) == 0:
            return 0.0

        return get_wins(historial, team) / len(historial)

    def comparar(
        win_rate_local: float,
        win_rate_visitante: float
    ) -> str:

        if win_rate_local > win_rate_visitante:
            return "L"
        elif win_rate_local < win_rate_visitante:
            return "V"
        return "E"

    def calcular_atributos_fila(
        row: pd.Series
    ) -> pd.Series:

        fecha = row["date"]
        equipo_local = row["home"]
        equipo_visitante = row["away"]

        historial_local = get_historial(
            fecha,
            equipo_local
        )
        historial_visitante = get_historial(
            fecha,
            equipo_visitante
        )

        tasa_local = get_win_rate(
            historial_local,
            equipo_local
        )
        tasa_visitante = get_win_rate(
            historial_visitante,
            equipo_visitante
        )

        ventaja_historica = comparar(
            tasa_local,
            tasa_visitante
        )

        ultimos_local = historial_local.tail(
            matches_limit
        )
        ultimos_visitante = historial_visitante.tail(
            matches_limit
        )

        condicion_local = get_win_rate(
            ultimos_local,
            equipo_local
        )
        condicion_visitante = get_win_rate(
            ultimos_visitante,
            equipo_visitante
        )

        condicion_partido = comparar(
            condicion_local,
            condicion_visitante
        )

        # historial_suficiente = int(
        #     len(ultimos_local) == matches_limit
        #     and len(ultimos_visitante) == matches_limit
        # )

        return pd.Series({
            "historial": ventaja_historica,
            "condition_match": condicion_partido,
            # "historial_suficiente": historial_suficiente
        })

    nuevos_atributos = dataset.apply(
        calcular_atributos_fila,
        axis=1
    )

    return pd.concat(
        [dataset, nuevos_atributos],
        axis=1
    )

In [23]:
from sklearn.preprocessing import OrdinalEncoder


df = load_dataset("futbol_uruguayo.csv")

df_procesado = load_attributes(
    df,
    years_limit=1,
    matches_limit=5
)


df_procesado = df_procesado.drop(columns=["date"])
cols_procesadas=["home", "away", "historial", "condition_match", "result"]  

enc = OrdinalEncoder(dtype=int)




# fit_transform aprende las categorías y transforma todo el bloque
df_procesado[cols_procesadas] = enc.fit_transform(df_procesado[cols_procesadas])

# for i, cat in enumerate(enc.categories_[0]):
#     print(f"{cat} -> {i}")

print(
    df_procesado[
        [
            #"date",
            "home",
            "away",
            "historial",
            "condition_match",
            #"historial_suficiente",
            "result"
        ]
    ].head(20)
)

    home  away  historial  condition_match  result
0      1    17          0                0       2
1      7    33          0                0       0
2     10    31          0                0       1
3     26    30          0                0       1
4     27    22          0                0       1
5      7    22          0                0       1
6     17    30          1                1       1
7     26    10          0                0       0
8     27    33          1                1       0
9     31     1          0                0       1
10     7    10          0                0       1
11    17    31          1                1       0
12    26    22          1                1       1
13    30    27          2                2       1
14    33     1          0                0       1
15     1    30          2                2       1
16     7    31          1                1       0
17    10    22          1                1       1
18    17    33          1      

In [26]:
for i, cat in enumerate(enc.categories_[0]):
    print(f"{cat} -> {i}")

Albion -> 0
Bella Vista -> 1
Boston River -> 2
CA Basanez -> 3
CA Cerro -> 4
CA Fenix -> 5
CA Juventud -> 6
CA Penarol -> 7
CA Progreso -> 8
CSyd Villa Espanola -> 9
Central Espanol -> 10
Cerro Largo FC -> 11
Club Atletico Atenas -> 12
Club Social Y Deportivo Huracan Buceo -> 13
Club Sportivo Cerrito -> 14
Colon FC -> 15
Danubio -> 16
Defensor Sporting -> 17
Dep Colonia -> 18
Deportivo Maldonado -> 19
El Tanque Sisley -> 20
Frontera Rivera Chico -> 21
Institucion Atletica Sud America -> 22
La Luz Football Club -> 23
Liverpool -> 24
Miramar Misiones -> 25
Montevideo Wanderers -> 26
Nacional -> 27
Paysandu FC -> 28
Plaza Colonia -> 29
Racing Club -> 30
Rampla Juniors Futbol Club -> 31
Rentistas -> 32
River Plate -> 33
Rocha Futbol Club -> 34
Tacuarembo Futbol Club -> 35
Torque FC -> 36
Villa Teresa -> 37


In [25]:
import sys
import os
import pandas as pd

# Agrega la carpeta padre (Tarea1) al path de búsqueda de Python
sys.path.append(os.path.abspath(".."))

# Ahora ya puedes importar el archivo o sus elementos
from decisionTree.clasifier import clasifier as DecisionTreeClassifier
# o importar funciones/clases específicas:
# from decisionTree.clasifier import MiClase, mi_funcion

dataset=pd.read_csv("entrenamiento_codificado.csv")

modelo = DecisionTreeClassifier(0.001)

atributos = ["historial", "condition_match"]
#atributos= ["ventaja_historica","ventaja_forma","ataque_local","defensa_local","ataque_visitante","defensa_visitante","ventaja_localia"]
# print(df_procesado.isna().sum())
# print(df_procesado.dtypes)
# print(df_procesado[df_procesado.isna().any(axis=1)].head())
# print(df_procesado.head())


modelo.fit(atributos, df_procesado)
modelo.tree.print_tree()



historial
  [0]
    1
  [1]
    condition_match
      [1]
        1
      [2]
        1
      [0]
        1
  [2]
    condition_match
      [2]
        2
      [1]
        1
      [0]
        1
